# AirFusion — Exploratory Data Analysis (EDA)
This notebook investigates multi-city air quality telemetry, pollutant distributions, diurnal trends, and atmospheric correlations.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.database import fetch_historical_records
from src.preprocessing import clean_air_quality_dataframe

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## 1. Load Data from SQLite Database

In [ ]:
df_raw = fetch_historical_records(city='Chandigarh')
print(f'Raw records fetched: {len(df_raw)}')
df_clean = clean_air_quality_dataframe(df_raw)
df_clean.head()

## 2. Summary Statistics & Data Types

In [ ]:
df_clean[['pm2_5', 'pm10', 'no2', 'o3', 'temperature', 'humidity', 'wind_speed']].describe()

## 3. Pollutant Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_clean['pm2_5'], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('Distribution of PM2.5 Concentrations (ug/m3)')
axes[0].axvline(55.5, color='red', linestyle='--', label='Unhealthy Threshold (55.5)')
axes[0].legend()

sns.boxplot(x=df_clean['pm2_5'], ax=axes[1], color='lightsteelblue')
axes[1].set_title('PM2.5 Outlier and Quartile Spread')
plt.tight_layout()
plt.show()

## 4. Hourly Diurnal Cycle (Rush Hour Peak Analysis)

In [ ]:
df_clean['hour'] = pd.to_datetime(df_clean['timestamp']).dt.hour
hourly_pm25 = df_clean.groupby('hour')['pm2_5'].agg(['mean', 'median', 'std']).reset_index()

plt.figure(figsize=(12, 5))
plt.plot(hourly_pm25['hour'], hourly_pm25['mean'], marker='o', color='darkred', label='Mean PM2.5')
plt.plot(hourly_pm25['hour'], hourly_pm25['median'], marker='s', color='navy', linestyle='--', label='Median PM2.5')
plt.fill_between(hourly_pm25['hour'], hourly_pm25['mean'] - hourly_pm25['std'], hourly_pm25['mean'] + hourly_pm25['std'], color='pink', alpha=0.3, label='+-1 Std Dev')
plt.title('Average PM2.5 Concentration by Hour of Day (Diurnal Cycle)')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('PM2.5 (ug/m3)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

## 5. Correlation Heatmap: Pollutants and Weather

In [ ]:
cols = ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2', 'temperature', 'humidity', 'wind_speed', 'pressure']
corr = df_clean[[c for c in cols if c in df_clean.columns]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
plt.title('Cross-Correlation Matrix: Pollutants and Meteorological Factors')
plt.tight_layout()
plt.show()